In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/Bengali Paraphrasing'
DATA_DIR = f'{PROJECT_ROOT}/BanglaParaphrase'
PROCESSED_DIR = f'{PROJECT_ROOT}/processed_data'

os.makedirs(PROCESSED_DIR, exist_ok=True)

files = {
    'train': os.path.join(DATA_DIR, 'train.jsonl'),
    'validation': os.path.join(DATA_DIR, 'validation.jsonl'),
    'test': os.path.join(DATA_DIR, 'test.jsonl'),
}

Mounted at /content/drive


# Load raw data

In [2]:
import json
import pandas as pd

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

dfs = {split: load_jsonl(path) for split, path in files.items()}
for split, df in dfs.items():
    print(f"{split}: {len(df)} rows loaded")

train: 419967 rows loaded
validation: 23331 rows loaded
test: 23332 rows loaded


# Clean the data
### Since there are near-zero duplication and clean samples, so cleaning here is light-touch — just standard hygiene, not heavy filtering!

In [3]:
def clean_df(df):
    before = len(df)

    # Drop rows with missing source/target
    df = df.dropna(subset=['source', 'target'])

    # Strip leading/trailing whitespace
    df['source'] = df['source'].astype(str).str.strip()
    df['target'] = df['target'].astype(str).str.strip()

    # Drop rows where source or target became empty after stripping
    df = df[(df['source'].str.len() > 0) & (df['target'].str.len() > 0)]

    # Drop exact duplicate (source, target) pairs -- confirmed only 4 in train, 0 elsewhere
    df = df.drop_duplicates(subset=['source', 'target'])

    # Reset index
    df = df.reset_index(drop=True)

    after = len(df)
    print(f"  Before: {before} -> After: {after} (removed {before - after})")
    return df

cleaned_dfs = {}
for split, df in dfs.items():
    print(f"Cleaning {split}:")
    cleaned_dfs[split] = clean_df(df)

Cleaning train:
  Before: 419967 -> After: 419963 (removed 4)
Cleaning validation:
  Before: 23331 -> After: 23331 (removed 0)
Cleaning test:
  Before: 23332 -> After: 23332 (removed 0)


# Save cleaned data
### We're using Parquet rather than CSV — it preserves Bengali Unicode text reliably without encoding issues, loads faster, and is smaller on disk, which matters since we're sharing this across multiple Colab sessions via Drive.

In [4]:
for split, df in cleaned_dfs.items():
    out_path = os.path.join(PROCESSED_DIR, f'{split}_clean.parquet')
    df.to_parquet(out_path, index=False)
    print(f"Saved {split} -> {out_path} ({len(df)} rows)")

Saved train -> /content/drive/MyDrive/Bengali Paraphrasing/processed_data/train_clean.parquet (419963 rows)
Saved validation -> /content/drive/MyDrive/Bengali Paraphrasing/processed_data/validation_clean.parquet (23331 rows)
Saved test -> /content/drive/MyDrive/Bengali Paraphrasing/processed_data/test_clean.parquet (23332 rows)


# Save a small metadata file (for reproducibility / paper documentation)

In [5]:
metadata = {
    'source_dataset': 'BanglaParaphrase (Akil et al., 2022)',
    'license': 'CC BY-NC-SA 4.0',
    'split_sizes': {split: len(df) for split, df in cleaned_dfs.items()},
    'cleaning_steps': [
        'dropped rows with missing source/target',
        'stripped whitespace',
        'dropped rows empty after stripping',
        'dropped exact duplicate (source, target) pairs'
    ]
}

import json as json_lib
with open(os.path.join(PROCESSED_DIR, 'metadata.json'), 'w', encoding='utf-8') as f:
    json_lib.dump(metadata, f, ensure_ascii=False, indent=2)

print(json_lib.dumps(metadata, ensure_ascii=False, indent=2))

{
  "source_dataset": "BanglaParaphrase (Akil et al., 2022)",
  "license": "CC BY-NC-SA 4.0",
  "split_sizes": {
    "train": 419963,
    "validation": 23331,
    "test": 23332
  },
  "cleaning_steps": [
    "dropped rows with missing source/target",
    "stripped whitespace",
    "dropped rows empty after stripping",
    "dropped exact duplicate (source, target) pairs"
  ]
}
